In [11]:
# perform reading here
import os 
import sys

from docling.document_converter import DocumentConverter
import pandas as pd
import docx2txt

from pathlib import Path

Reading documents

In [12]:
input_folder_path = 'src/ffma/qdev/synth_ai/ingestion/docs/ic_mrc_arc_large_batch'  # Folder containing the .pdf and .docx files
output_folder_path = 'src/ffma/qdev/synth_ai/ingestion/docs/reader-outputs/ic_mrc_arc_large_batch'

# converter = DocumentConverter()
# for filename in os.listdir(input_folder_path):
#     file_path = os.path.join(input_folder_path, filename)

#     # Construct the output file path with .txt extension
#     output_filepath = os.path.join(output_folder_path, filename.rsplit('.', 1)[0] + ".txt")

#     # Check if the .txt file already exists in the output folder
#     if os.path.exists(output_filepath):
#         print(f"Skipping {filename}, because the corresponding .txt file already exists.")
#         continue  # Skip this file and continue to the next one
#     if filename.endswith('.pdf'):
#         print(f"Reading PDF: {filename}")
#         input_filepath = os.path.join(input_folder_path, filename)
#         output_filepath = os.path.join(output_folder_path, filename.replace(".pdf", ".txt"))

#         result = converter.convert(input_filepath)

#         print(f"Writing PDF: {filename}")
#         with open(output_filepath, "w", encoding="utf-8") as f_write:
#             if result:
#                 f_write.write(result.document.export_to_markdown())
#             else:
#                 raise ValueError(f"empty read from file: {filename}")
#     elif filename.endswith('.docx'):
#         print(f"Reading docx: {filename}")
#         input_filepath = os.path.join(input_folder_path, filename)
#         output_filepath = os.path.join(output_folder_path, filename.replace(".docx", ".txt"))

#         result = None
#         try:
#             result = docx(input_filepath)
#             print("DOCX loaded successfully!")
#         except Exception as e:
#             print(f"Failed to read as DOCX: {e}")

#         print(f"Writing docx: {filename}")
#         with open(output_filepath, "w", encoding="utf-8") as f_write:
#             if result:
#                 f_write.write(result.document.export_to_markdown())
#             else:
#                 raise ValueError(f"empty read from file: {filename}")
#                     print(f"Writing docx: {filename}")
#     elif filename.endswith('.xlsx'):
#         print(f"Reading xlsx: {filename}")
#         input_filepath = os.path.join(input_folder_path, filename)
#         output_filepath = os.path.join(output_folder_path, filename.replace(".xlsx", ".txt"))

#         df, list_results = None, None
#         with open(input_filepath, "rb") as f:
#             result = pd.read_excel(input_filepath)
#             list_results = result.applymap(str).values.flatten().tolist()

#         print(f"Writing xlsx: {filename}")
#         with open(output_filepath, "w", encoding="utf-8") as f_write:
#             if list_results is not None:
#                 f_write.write(" ".join(list_results)
#             else:
#                 raise ValueError(f"empty read from file: {filename}")
#     else:
#         raise NotImplementedError(f"File extension not supported: {filename}")
#         continue

In [14]:
result = ''
with open('../data/raw/files/ARC - 20240620 - Minutes (FINAL) (A1179933).docx', 'rb') as file:
    result = docx2txt.process(file)

ARC Meeting - Minutes

Asset Review Committee (ARC) Meeting

Date:	

Thursday 20 June 2024

Time:	

02.00 pm to 03.00 pm

Members:

Tammi Fisher (Chair), Belinda Chain, Belinda Cheung, James Fraser-Smith, Maria Guo, Kelvin Mak-Lui and James Waldron

Apologies:

Scott Day and James White

Other attendees:

Wai-Kwan Chislett (Secretary) 

Observers: Deniz, Alev, Lucy O’Malley

Presenters:

Item 2.1: David Bluff, Chris Burton, Karel Tan and Stephen Siu



Item

Items for discussion/approval

1.

ARC administration

1.1

Attendance and apologies



The Chair noted that a quorum was present.

 1.2

Conflict of interest declarations



The Chair confirmed with the Committee members that none of them had any material personal interest, or other real or perceived conflict of interest, in any of the items to be considered at the meeting that may preclude their involvement.

1.3

			Minutes and action items from previous meeting 

			

			The Committee approved the minutes of the Tuesday 7 May 2

In [18]:
def construct_output_filepath(input_folder, output_folder, filename, extension):
    """Constructs the output file path with the specified extension."""
    return os.path.join(output_folder, filename.rsplit('.', 1)[0] + extension)

def file_already_processed(output_filepath):
    """Checks if the output file already exists."""
    if os.path.exists(output_filepath):
        print(f"Skipping {os.path.basename(output_filepath)}, because the corresponding .txt file already exists.")
        return True
    return False

def read_and_write_file(input_filepath, output_filepath, content_extractor):
    """Reads a file, extracts content, and writes it to an output file."""
    result = content_extractor(input_filepath)
    print(f"Writing {os.path.basename(output_filepath)}")
    with open(output_filepath, "w", encoding="utf-8") as f_write:
        if result:
            f_write.write(result)
        else:
            raise ValueError(f"Empty read from file: {os.path.basename(input_filepath)}")

def process_pdf(converter, input_filepath, output_filepath):
    print(f"Reading PDF: {os.path.basename(input_filepath)}")
    result = converter.convert(input_filepath)
    return result.document.export_to_markdown() if result else None

def process_docx(input_filepath, output_filepath):
    print(f"Reading DOCX: {os.path.basename(input_filepath)}")
    try:
        result = None
        with open(input_filepath, 'rb') as file:
            result = docx2txt.process(file)
        print("DOCX loaded successfully!")
        return result if result else None
    except Exception as e:
        print(f"Failed to read as DOCX: {e}")
        return None

def process_xlsx(input_filepath, output_filepath):
    print(f"Reading XLSX: {os.path.basename(input_filepath)}")
    df = pd.read_excel(input_filepath)
    list_results = df.applymap(str).values.flatten().tolist()
    return " ".join(list_results) if list_results else None

def process_files(input_folder, output_folder):
    converter = DocumentConverter()
    for filename in os.listdir(input_folder):
        input_filepath = os.path.join(input_folder, filename)
        
        if filename.endswith('.pdf'):
            output_filepath = construct_output_filepath(input_folder, output_folder, filename, ".txt")
            if file_already_processed(output_filepath):
                continue
            read_and_write_file(input_filepath, output_filepath, lambda f: process_pdf(converter, f, output_filepath))

        elif filename.endswith('.docx'):
            output_filepath = construct_output_filepath(input_folder, output_folder, filename, ".txt")
            if file_already_processed(output_filepath):
                continue
            read_and_write_file(input_filepath, output_filepath, lambda f: process_docx(f, output_filepath))

        elif filename.endswith('.xlsx'):
            output_filepath = construct_output_filepath(input_folder, output_folder, filename, ".txt")
            if file_already_processed(output_filepath):
                continue
            read_and_write_file(input_filepath, output_filepath, lambda f: process_xlsx(f, output_filepath))
        
        else:
            raise NotImplementedError(f"File extension not supported: {filename}")

In [20]:
input_folder = "../data/raw/files/miniTestSubset/"
output_folder = "../data/processed/pre_processing/ocr_output"

process_files(input_folder, output_folder)

Skipping 01 ARC - Decision item register 2024 (A1136154).txt, because the corresponding .txt file already exists.
Skipping ARC - 20240507 - Papers (A1164349).txt, because the corresponding .txt file already exists.
Reading DOCX: ARC - 20240620 - Minutes (FINAL) (A1179933).docx
DOCX loaded successfully!
Writing ARC - 20240620 - Minutes (FINAL) (A1179933).txt


Extracting entities from documents

In [ ]:
from src

In [ ]:
STATIC_MODEL = "gpt-35-turbo"

LIMITS = {
    "gpt-35-turbo": {
        "max_tokens_no_chunk": 4000,
        "max_tokens_prompt": 1000,
        "max_tokens_per_chunk": 2000,
    },
    "gpt-4": {
        "max_tokens_no_chunk": 8000,
        "max_tokens_prompt": 2000,
        "max_tokens_per_chunk": 4000,
    },
}

In [ ]:
async def read_txt_file(file_path: str) -> str:
    try:
        async with aiofiles.open(file_path, mode="r", encoding="utf-8") as file:
            content = await file.read()
        return content
    except Exception as e:
        print(f"Error reading file: {e}")
        return ""

In [ ]:
from src.utils.helpers import clean_text, get_azure_openai_client

def openai_completion(question: dict, model: str, context: str = None, chunks: list = None) -> str:
    client = get_azure_openai_client().get("client")
    question_prompt = question["prompt"]

    if chunks:
        completions = []

        for chunk in chunks:
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "system",
                        "content": "You are an assistant that only uses the provided context to answer questions. If the context does not include the answer, respond with an empty string: """
                    },
                    {
                        "role": "user",
                        "content": (
                            f"Based on the following context, answer the question: \n\n"
                            f"Context: {chunk} \n\n"
                            f"Question: {question_prompt} in the format described, and your response should only contain the answer, without any parts of the question."
                        )
                    }
                ],
                temperature=0.0
            )

            completions.append(clean_text(completion.choices[0].message.content or ""))

        completion_text = " ".join(completions)
        final_completion = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": """You are an assistant that only uses the provided context to answer questions. If the context is empty, respond with an empty string: "". 
                    If you are unable to answer the question, respond with an empty string. Enclose all responses into a list like so: ["answer1", "answer2", ""].
                    Remove "[" and "]" characters in entries inside the list."""
                },
                {
                    "role": "user",
                    "content": """The following context contains answers to the original question:
                    Original Question: {question_prompt} asked to different chunks of a document. Based on the context, answer the final question.
                    Context: {completion_text}
                    Final Question: Compile the context into a single word or few words, without losing the central idea, in the format described in the original question.
                    Respond without any parts of the original question and without empty strings in the response unless the context is empty."""
                }
            ],
            temperature=0.0
        )

        return clean_text(final_completion.choices[0].message.content or "")

    elif context:
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": """You are an assistant that only uses the provided context to answer questions. If the context does not include the answer, respond with an empty string. 
                    Enclose all responses into a list like so: ["answer1", "answer2", ""]. Remove "[" and "]" characters in entries inside the list."""
                },
                {
                    "role": "user",
                    "content": (
                        f"Based on the following context, answer the question: \n\n"
                        f"Context: {context} \n\n"
                        f"Question: {question_prompt}. Your response should be a single word or few words, "
                        f"in the format described, and your response should only contain the answer, without any parts of the question, "
                        f"and without any nulls in the response unless the context is empty."
                    )
                }
            ],
            temperature=0.0
        )

        return clean_text(completion.choices[0].message.content or "")

    else:
        return 'ERROR: No context or chunks provided'

In [ ]:
# perform entity extraction here with LLM
async def ee_openai_runner(file_path: str, prompts: dict) -> dict:

    #prebuiltLayoutRunner removed for now, reading text files instead
    context = await read_txt_file(file_path)
    responses = {}

    model = STATIC_MODEL
    if len(context) > LIMITS[STATIC_MODEL]["max_tokens_no_chunk"]:
        model = EXPLORATORY_MODEL
        if len(context) > LIMITS[model]["max_tokens_no_chunk"]:
            chunks = chunk_text(context, tiktoken.encoding_for_model(model), LIMITS[model]["max_tokens_per_chunk"])
            for prompt_name, prompt in prompts.items():
                response = openai_completion(prompt, model, chunks=chunks)
                responses[prompt_name] = response
            return responses

    for prompt_name, prompt in prompts.items():
        response = openai_completion(prompt, model, context=context)
        responses[prompt_name] = response

    return responses

In [ ]:
def list_files_in_folder(root_folder: str):
    file_list = glob.glob(os.path.join(root_folder, "**", "*"), recursive=True)
    return [f for f in file_list if os.path.isfile(f)]  # Filter out directories

In [ ]:
 list_files = list_files_in_folder("docs/ic_mrc_arc")

    for file_path in list_files:
        base_name = Path(file_path).stem
        print(f"processing document: {file_path}")

        responses = asyncio.run(ee_openai_runner(file_path, ic_mrc_arc_prompts))
        print(responses)

        if not isinstance(responses, dict):
            print("Responses are not an object")
            return

        cleaned_responses = clean_json(responses)
        json_responses = json.dumps(cleaned_responses, indent=2)

        timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
        output_dir = Path("./outputs/ee/openai/ic_mrc_arc")
        output_dir.mkdir(parents=True, exist_ok=True)

        output_file = output_dir / f"response-{base_name}-{timestamp}.json"
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(json_responses)

        print(f"Output file: {output_file}")

In [ ]:
# then clean entities with pre_process_data.py